# Connector: serving targets

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pecca-core/pecca/blob/main/examples/notebooks/connector_serving.ipynb)

`onnx_local` serves in-process (what `@pecca.replace` uses); `docker_export` writes a folder you can build into a container; `databricks_serving` creates a Model Serving endpoint. This notebook benchmarks local serving and shows the Docker export. It uses the tiny hashing embedder (`PECCA_TEST_SMALL=1`) so nothing is downloaded.

In [1]:
%pip install -q pecca

In [2]:
import json, os, re, shutil, subprocess, tempfile, time
from pathlib import Path

import pandas as pd
import pecca
from pecca.core import context
from pecca.data.dataset import Dataset, canonicalize
from pecca.data.synthetic import generate

HERE = Path.cwd()                                   # the notebook's folder (examples/notebooks in the repo)
work = tempfile.mkdtemp()
os.chdir(work)
os.environ["PECCA_HOME"] = os.path.join(work, ".pecca")
os.environ["TQDM_DISABLE"] = "1"
os.environ["MLFLOW_SUPPRESS_PRINTING_URL_TO_STDOUT"] = "1"
import logging
try:
    import mlflow  # MLflow resets its logger level on import, so quiet it afterwards
    logging.getLogger("mlflow").setLevel(logging.ERROR)
except ImportError:
    pass

# Docker-backed cells run only when Docker is reachable and PECCA_SKIP_DOCKER != 1 (CI sets it).
USE_DOCKER = (os.environ.get("PECCA_SKIP_DOCKER") != "1" and shutil.which("docker") is not None
              and subprocess.run(["docker", "info"], capture_output=True).returncode == 0)
print("docker:", "available" if USE_DOCKER else "skipped (PECCA_SKIP_DOCKER=1 or Docker not reachable)")

def sh(*args, check=True):
    return subprocess.run([str(a) for a in args], capture_output=True, text=True, check=check)

import atexit
def cleanup_later(*cmd):
    """Run `cmd` when the kernel exits, so containers are removed even if a cell fails."""
    atexit.register(lambda: subprocess.run([str(c) for c in cmd], capture_output=True))

def free_port():
    import socket
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]

def wait_for(check, what, seconds=90):
    end = time.time() + seconds
    while time.time() < end:
        try:
            if check():
                return
        except Exception:
            pass
        time.sleep(1)
    raise TimeoutError(f"{what} did not become ready in {seconds}s")

def train_demo(call="route_rfi", rows=3000, candidates=("tfidf_linear",), **kw):
    """Train a small model on the synthetic demo data (LLM answers + human corrections)."""
    df = generate(rows)
    df["text"] = df.email_subject + "\n\n" + df.email_body
    cols = {"input": "text", "llm_output": "llm_rfi", "human_label": "human_rfi", "call_name": {"literal": call.split("/")[-1]}}
    return pecca.train(call, Dataset(canonicalize(df, cols), cols), candidates=list(candidates), **kw)

docker: available


In [3]:
os.environ["PECCA_TEST_SMALL"] = "1"
r1 = train_demo(candidates=("tfidf_linear",))
r2 = train_demo(candidates=("e5_logreg",))
call = context.get_call("route_rfi")
print({m.version: (m.candidate, m.format) for m in call.versions()})

{'v1': ('tfidf_linear', 'pickle'), 'v2': ('e5_logreg', 'onnx')}


## Latency of in-process serving (single-row predictions on CPU)

In [4]:
import numpy as np
texts = (generate(200, seed=3).email_subject + "\n\n" + generate(200, seed=3).email_body).tolist()
from pecca.runtime.predictor import get_predictor
bench = {}
for version in ("v1", "v2"):
    p = get_predictor(call, version)
    p.predict(texts[:1])                                   # warm-up / lazy load
    times = []
    for t in texts:
        t0 = time.perf_counter(); p.predict([t]); times.append((time.perf_counter() - t0) * 1000)
    m = call.version(version)
    bench[f"{version} {m.candidate} ({m.format})"] = {"median ms": round(float(np.median(times)), 2), "p95 ms": round(float(np.percentile(times, 95)), 2)}
pd.DataFrame(bench).T

,median ms,p95 ms
v1 tfidf_linear (pickle),0.38,0.41
v2 e5_logreg (onnx),0.20,0.22


## docker_export
Writes a Dockerfile, a FastAPI `/predict` app, `requirements.txt` and the model for one version.

In [5]:
from pecca import connectors
export = connectors.create("serving", {"type": "docker_export", "out_dir": "export", "home": os.environ["PECCA_HOME"]})
folder = Path(export.deploy(call.key, "v2"))
print(sorted(str(p.relative_to(folder)) for p in folder.rglob("*") if p.is_file()))
print((folder / "Dockerfile").read_text())
print((folder / "requirements.txt").read_text())

['Dockerfile', 'app.py', 'model/calibration.json', 'model/candidate.json', 'model/config.json', 'model/labels.json', 'model/lineage.json', 'model/metadata.json', 'model/model.onnx', 'model/model.pkl', 'requirements.txt']
FROM python:3.11-slim
WORKDIR /srv
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]

pecca==0.1.0
fastapi
uvicorn



In [6]:
print((folder / "app.py").read_text())
print("# build and run it (skipped here: the image installs torch via pecca, which is large):")
print(f"docker build -t pecca-route-rfi {folder.name} && docker run -p 8000:8000 pecca-route-rfi")
print('curl -s localhost:8000/predict -H "content-type: application/json" -d \'{"inputs": ["Card lost\\n\\nplease block my card"]}\'')

from fastapi import FastAPI
from pydantic import BaseModel

from pecca.candidates import get_candidate
import json, pathlib

app = FastAPI(title="pecca-route_rfi")
MODEL_DIR = pathlib.Path(__file__).parent / "model"
cfg = json.loads((MODEL_DIR / "config.json").read_text())
labels = json.loads((MODEL_DIR / "labels.json").read_text())
cand = get_candidate(cfg["candidate"]).load(str(MODEL_DIR))


class Req(BaseModel):
    inputs: list[str]


@app.post("/predict")
def predict(req: Req):
    proba = cand.predict_proba(req.inputs)
    classes = cand.classes_
    return [{"label": labels["forms"].get(classes[int(p.argmax())], classes[int(p.argmax())]),
              "confidence": float(p.max())} for p in proba]

# build and run it (skipped here: the image installs torch via pecca, which is large):
docker build -t pecca-route-rfi route_rfi-v2 && docker run -p 8000:8000 pecca-route-rfi
curl -s localhost:8000/predict -H "content-type: application/json" -d '{"inputs": ["Card lost\n\nplease block 